
# 🧰 Week 5 — PydanticAI Tool Calling (Important Concepts Only)

This notebook focuses only on the **most important concepts** for a 30–50 min live session:

1. `@agent.tool_plain`
2. `@agent.tool`
3. `RunContext + deps`
4. Multi-tool chaining
5. Error handling

---

## Goal
Understand how AI agents use Python functions as tools.


In [43]:

# Install dependencies
%pip install pydantic-ai python-dotenv groq rich --quiet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [44]:

# Setup

import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("GROQ_API_KEY"), "❌ GROQ_API_KEY not found in .env"

print("✅ Environment ready")


✅ Environment ready



# 1️⃣ Your First Tool — `@agent.tool_plain`

This is the MOST important concept.

- Tool = normal Python function
- Agent decides when to call it
- Tool returns result back to LLM


In [45]:

import math
from pydantic_ai import Agent
# agent = ai assistant that can do calculations. It should always use the calculation tool for arithmetic problems.

# create an ai agent 
#  this agent behave like math assistant. It should always use the calculation tool for arithmetic problems.
calc_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt=(
        "You are a math assistant. "
        "Always use the calculation tool for arithmetic."
    ),
)


# Register a tool using  @tool_plain decorator. This tool will safely evaluate math expressions.
# python function that ai can call automatically
# example "17* 5 + sqrt(16)" -> "17* 5 + sqrt(16)" -> 89.0
@calc_agent.tool_plain
def run_calculation(expression: str) -> str:
    '''
    Safely evaluate math expressions.
    '''
    safe_env = {
        k: getattr(math, k) # k = func name from math module
        for k in dir(math)  # loop through all functions in math module
        if not k.startswith("_") # ignore private functions
    }

    try:
        # eval() = execute the expression in a safe environment with only math functions
        # "__builtins__": {} = block dengrous python excess
        # safe_env = allow only math functions

        result = eval(expression, {"__builtins__": {}}, safe_env) # evaluate the expression with safe environment
        return str(result)
        
    except Exception as e:
        return f"Error: {e}"


In [46]:
result = await calc_agent.run(
    "What is 17 * 34 + sqrt(256)?"
)

print(result.output)

The result of the calculation 17 * 34 + sqrt(256) is 594.0.



# 2️⃣ Tools with Context (`RunContext + deps`)

Use this when:
- user roles matter
- authentication is needed
- database/session info is required

This is a VERY important real-world concept.


In [47]:
# 2️⃣ Tools with Context (`RunContext + deps`)
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext

@dataclass
class UserContext:
    username: str
    is_admin: bool



In [48]:

secure_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    deps_type=UserContext,
    system_prompt="You are a secure assistant.",
)

@secure_agent.tool
def get_secret(ctx: RunContext[UserContext]) -> str:
    # ctx contain runtime info about the agent execution, including deps (dependencies)

    
    if ctx.deps.is_admin: # check if user is admin
        return "✅ Admin Secret: Production access granted."
    
    return "❌ Access denied. Admin only."

In [49]:
# create a user  data with username and is_admin flag. This will be passed as deps to the agent when calling the tool.
user_ctx = UserContext(
    username="Nidhi",
    is_admin=False
)

result = await secure_agent.run(
    "Can I access the production secret?",
    deps=user_ctx
)

print(result.output)

I'm sorry, but I cannot access the production secret. If you need assistance with anything else, please let me know.



# 3️⃣ Multi-Tool Chaining

Agent can:
1. Call tool 1
2. Use its output
3. Call tool 2
4. Generate final answer

This makes agents powerful.


In [50]:
# multi tool
from pydantic_ai import Agent

multi_tool_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt=(
        "You are a smart assistant with weather and calculator tools."
    ),
)

@multi_tool_agent.tool_plain
def fetch_weather(city: str) -> str:
    '''
    Fake weather API for demo.
    '''
    weather_data = {
        "Tokyo": 28,
        "Delhi": 39,
        "London": 18
    }

    temp = weather_data.get(city, 25)

    return f"{city} temperature is {temp}°C"

@multi_tool_agent.tool_plain
def convert_to_fahrenheit(celsius: float) -> str:
    f = (celsius * 9/5) + 32
    return f"{f}°F"



In [51]:
result = await multi_tool_agent.run(
    "What is the temperature in Tokyo? "
    "Convert it to Fahrenheit."
)

print(result.output)

The current temperature in Tokyo is 28°C, which is equivalent to 71.6°F in Fahrenheit.



# 4️⃣ Tool Error Handling

Tools can fail.

Good agents should:
- catch errors
- return meaningful messages
- continue gracefully


In [52]:
# tool error handling
error_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt="Use tools when required.",
)

@error_agent.tool_plain
def divide(value: int) -> str:

    if value == 0:
        return "❌ Cannot divide by zero"

    return f"100 / {value} = {100 / value}"


In [53]:
result = await error_agent.run(
    "Divide 100 by 5 and also divide 100 by 0."
)

print(result.output)



<function=divide>{"value": 100 / 5}</function>




# ✅ Final Summary

| Concept | Purpose |
|---|---|
| `@agent.tool_plain` | Simple tools |
| `@agent.tool` | Tools with context |
| `RunContext` | Access deps/session |
| Multi-tool chaining | Combine multiple tools |
| Error handling | Safe execution |

---

# 🎯 Key Takeaway

AI agents become powerful when they can:
- think
- call tools
- use results
- continue reasoning
